# Hermes Agent on Google Colab

Google Colab の A100 GPU 上で、Ollama の `gemma4:31b` を使って Hermes Agent を動かします。
Hermes の記憶・設定・SQLite・skills と作業ファイルは、各ターン後に Google Drive へ同期します。

**対象:** Colab Pro / Pro+ 等で A100 80 GB 以上を利用できる方

**前提:**

- ランタイムのハードウェア アクセラレータを **A100 GPU** に設定済みであること
- Google Drive に少なくとも Agent 状態と作業ファイルを保存できる空きがあること
- モデル約 20 GB はランタイムごとに取得するため、十分なランタイムディスクと通信量があること

**重要:** セルは上から順に実行してください。`allow_dangerous=True` は、その呼び出しに限って
Hermes の危険操作承認を省略します。信頼できる指示だけに使用してください。


## 実行手順

1. 定数と保存先を初期化する
2. Google Drive をマウントし、A100・RAM・ディスクを検査する
3. 永続状態を Drive からローカルへ復元する
4. Hermes Agent と Ollama を導入・設定する
5. `gemma4:31b` を取得して疎通確認する
6. `ask_hermes()` で会話し、各ターン後に自動保存する


## 1. 定数とローカル保存先

SQLite は Google Drive の FUSE 上ではなく `/content` で動かします。
DriveへはSQLite Backup APIで作成した正常なコピーだけを保存します。


In [ ]:
from __future__ import annotations

import atexit
import importlib.metadata
import json
import os
import re
import shutil
import signal
import sqlite3
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

MODEL = "gemma4:31b"
HERMES_VERSION = "0.17.0"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_CONTEXT_LENGTH = "65536"

LOCAL_HOME = Path("/content/hermes-home")
LOCAL_WORKSPACE = Path("/content/hermes-workspace")
LOCAL_OLLAMA_MODELS = Path("/content/ollama-models")
OLLAMA_LOG = Path("/content/ollama.log")

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "Hermes"
DRIVE_STATE = DRIVE_ROOT / "state"
DRIVE_WORKSPACE = DRIVE_ROOT / "workspace"
DRIVE_SNAPSHOTS = DRIVE_ROOT / "snapshots"

RUNTIME_STATE_FILE = LOCAL_HOME / "colab_runtime.json"
CONFIG_FILE = LOCAL_HOME / "config.yaml"

# Hermes must see this before any Hermes module or subprocess starts.
os.environ["HERMES_HOME"] = str(LOCAL_HOME)
os.environ["TERMINAL_CWD"] = str(LOCAL_WORKSPACE)
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_CONTEXT_LENGTH"] = OLLAMA_CONTEXT_LENGTH
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MODELS"] = str(LOCAL_OLLAMA_MODELS)

for directory in (LOCAL_HOME, LOCAL_WORKSPACE, LOCAL_OLLAMA_MODELS):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Hermes home: {LOCAL_HOME}")
print(f"Workspace:   {LOCAL_WORKSPACE}")
print(f"Drive root:  {DRIVE_ROOT}")


## 2. Driveマウントとハードウェア検査

条件不足の場合はモデルを勝手に変更せず、ここで停止します。A100の割り当てはColab側で変更してください。


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("このNotebookはGoogle Colab上で実行してください。") from exc

drive.mount(str(DRIVE_MOUNT))

def _run_text(command: list[str]) -> str:
    return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT).strip()

try:
    gpu_line = _run_text([
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ]).splitlines()[0]
    gpu_name, gpu_memory_text = [part.strip() for part in gpu_line.rsplit(",", 1)]
    gpu_memory_mib = int(gpu_memory_text)
except (FileNotFoundError, subprocess.CalledProcessError, ValueError, IndexError) as exc:
    raise RuntimeError("NVIDIA GPUを検出できません。ColabのランタイムをA100 GPUへ変更してください。") from exc

ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_free_gib = shutil.disk_usage("/content").free / 1024**3

failures = []
if "A100" not in gpu_name.upper() or gpu_memory_mib < 39_000:
    failures.append(f"GPU: {gpu_name} ({gpu_memory_mib / 1024:.1f} GiB VRAM)")
if ram_gib < 24:
    failures.append(f"RAM: {ram_gib:.1f} GiB（24 GiB以上が必要）")
if disk_free_gib < 30:
    failures.append(f"空きディスク: {disk_free_gib:.1f} GiB（30 GiB以上が必要）")
if failures:
    raise RuntimeError("実行要件を満たしていません:\n- " + "\n- ".join(failures))

for directory in (DRIVE_ROOT, DRIVE_STATE, DRIVE_WORKSPACE, DRIVE_SNAPSHOTS):
    directory.mkdir(parents=True, exist_ok=True)

print(f"GPU:       {gpu_name} ({gpu_memory_mib / 1024:.1f} GiB VRAM)")
print(f"RAM:       {ram_gib:.1f} GiB")
print(f"Disk free: {disk_free_gib:.1f} GiB")
print("Hardware preflight: OK")


## 3. 復元・チェックポイント機能

`checkpoint()` はAgent状態を一旦ローカルに構築し、全SQLiteを検査してからDriveへ反映します。
`restore()` は現在状態が壊れていれば、新しい順に正常なスナップショットを探します。


In [ ]:
HOME_EXCLUDED_DIRS = {
    ".git", "__pycache__", "audio_cache", "backups", "bin", "cache",
    "hermes-agent", "image_cache", "logs", "node", "ollama",
}
HOME_EXCLUDED_FILES = {".env"}
WORKSPACE_EXCLUDED_DIRS = {".ipynb_checkpoints", "__pycache__"}

def _is_relative_to(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

def _assert_drive_target(path: Path) -> None:
    if not _is_relative_to(path, DRIVE_ROOT):
        raise ValueError(f"Drive操作対象がHermesディレクトリ外です: {path}")

def _sqlite_integrity(path: Path) -> tuple[bool, str]:
    try:
        connection = sqlite3.connect(f"file:{path}?mode=ro", uri=True, timeout=10)
        try:
            row = connection.execute("PRAGMA integrity_check").fetchone()
        finally:
            connection.close()
        result = str(row[0]) if row else "no result"
        return result.lower() == "ok", result
    except sqlite3.Error as exc:
        return False, str(exc)

def _backup_sqlite(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    src = sqlite3.connect(str(source), timeout=30)
    dst = sqlite3.connect(str(destination), timeout=30)
    try:
        src.backup(dst)
    finally:
        dst.close()
        src.close()
    ok, detail = _sqlite_integrity(destination)
    if not ok:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f"SQLiteバックアップの整合性検査に失敗しました: {source}: {detail}")

def _excluded(relative: Path, excluded_dirs: set[str], excluded_files: set[str]) -> bool:
    if any(part in excluded_dirs for part in relative.parts):
        return True
    if relative.name in excluded_files:
        return True
    return relative.name.endswith(("-wal", "-shm"))

def _copy_home_for_checkpoint(source_root: Path, destination_root: Path) -> None:
    destination_root.mkdir(parents=True, exist_ok=True)
    for source_path in source_root.rglob("*"):
        relative = source_path.relative_to(source_root)
        if _excluded(relative, HOME_EXCLUDED_DIRS, HOME_EXCLUDED_FILES):
            continue
        if source_path.is_symlink():
            continue
        destination_path = destination_root / relative
        if source_path.is_dir():
            destination_path.mkdir(parents=True, exist_ok=True)
        elif source_path.suffix == ".db":
            _backup_sqlite(source_path, destination_path)
        elif source_path.is_file():
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

def _validate_state_tree(root: Path) -> tuple[bool, str]:
    for database in root.rglob("*.db"):
        ok, detail = _sqlite_integrity(database)
        if not ok:
            return False, f"{database}: {detail}"
    return True, "ok"

def _replace_drive_directory(source: Path, destination: Path) -> None:
    _assert_drive_target(destination)
    token = uuid.uuid4().hex[:10]
    pending = DRIVE_ROOT / f".{destination.name}-next-{token}"
    previous = DRIVE_ROOT / f".{destination.name}-previous-{token}"
    _assert_drive_target(pending)
    _assert_drive_target(previous)
    shutil.rmtree(pending, ignore_errors=True)
    shutil.copytree(source, pending)
    if destination.exists():
        destination.rename(previous)
    pending.rename(destination)
    shutil.rmtree(previous, ignore_errors=True)

def _copy_tree(source: Path, destination: Path, excluded_dirs: set[str] | None = None) -> None:
    excluded_dirs = excluded_dirs or set()
    destination.mkdir(parents=True, exist_ok=True)
    for source_path in source.rglob("*"):
        relative = source_path.relative_to(source)
        if any(part in excluded_dirs for part in relative.parts) or source_path.is_symlink():
            continue
        destination_path = destination / relative
        if source_path.is_dir():
            destination_path.mkdir(parents=True, exist_ok=True)
        elif source_path.is_file():
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

def _snapshot_candidates() -> list[Path]:
    if not DRIVE_SNAPSHOTS.exists():
        return []
    pattern = re.compile(r"^\d{8}T\d{6}Z-[0-9a-f]{6}$")
    return sorted(
        (path for path in DRIVE_SNAPSHOTS.iterdir() if path.is_dir() and pattern.match(path.name)),
        reverse=True,
    )

def _prune_snapshots(keep: int = 3) -> None:
    for old_snapshot in _snapshot_candidates()[keep:]:
        _assert_drive_target(old_snapshot)
        shutil.rmtree(old_snapshot)

def checkpoint() -> Path:
    """Persist Hermes state and workspace to Drive; keep three valid state snapshots."""
    if not DRIVE_MOUNT.exists() or not DRIVE_ROOT.exists():
        raise RuntimeError("Google Driveがマウントされていません。")
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    snapshot_name = f"{timestamp}-{uuid.uuid4().hex[:6]}"
    with tempfile.TemporaryDirectory(prefix="hermes-checkpoint-", dir="/content") as temp_dir:
        local_snapshot = Path(temp_dir) / "state"
        _copy_home_for_checkpoint(LOCAL_HOME, local_snapshot)
        ok, detail = _validate_state_tree(local_snapshot)
        if not ok:
            raise RuntimeError(f"チェックポイントを中止しました: {detail}")

        drive_pending = DRIVE_SNAPSHOTS / f".{snapshot_name}.pending"
        drive_snapshot = DRIVE_SNAPSHOTS / snapshot_name
        _assert_drive_target(drive_pending)
        shutil.rmtree(drive_pending, ignore_errors=True)
        shutil.copytree(local_snapshot, drive_pending)
        ok, detail = _validate_state_tree(drive_pending)
        if not ok:
            shutil.rmtree(drive_pending, ignore_errors=True)
            raise RuntimeError(f"Driveへの書き込み検査に失敗しました: {detail}")
        drive_pending.rename(drive_snapshot)
        _replace_drive_directory(local_snapshot, DRIVE_STATE)

    with tempfile.TemporaryDirectory(prefix="hermes-workspace-", dir="/content") as temp_dir:
        workspace_snapshot = Path(temp_dir) / "workspace"
        _copy_tree(LOCAL_WORKSPACE, workspace_snapshot, WORKSPACE_EXCLUDED_DIRS)
        _replace_drive_directory(workspace_snapshot, DRIVE_WORKSPACE)

    _prune_snapshots(keep=3)
    print(f"Checkpoint saved: {drive_snapshot.name}")
    return drive_snapshot

def _select_restore_state() -> Path | None:
    candidates = [DRIVE_STATE, *_snapshot_candidates()]
    for candidate in candidates:
        if not candidate.exists() or not any(candidate.iterdir()):
            continue
        ok, detail = _validate_state_tree(candidate)
        if ok:
            return candidate
        print(f"Skipping invalid state {candidate.name}: {detail}")
    return None

def restore(force: bool = False) -> Path | None:
    """Restore the newest valid state and workspace. Use force=True to replace local data."""
    if not DRIVE_ROOT.exists():
        raise RuntimeError("Google DriveのHermesディレクトリが見つかりません。")
    state_source = _select_restore_state()
    local_has_state = any(LOCAL_HOME.iterdir())
    if state_source is not None and (force or not local_has_state):
        if force:
            shutil.rmtree(LOCAL_HOME)
            LOCAL_HOME.mkdir(parents=True, exist_ok=True)
        _copy_tree(state_source, LOCAL_HOME)
        print(f"State restored from: {state_source}")
    elif state_source is None:
        print("No saved Hermes state found; starting fresh.")
    else:
        print("Local Hermes state already exists; restore skipped.")

    local_has_workspace = any(LOCAL_WORKSPACE.iterdir())
    if DRIVE_WORKSPACE.exists() and any(DRIVE_WORKSPACE.iterdir()) and (force or not local_has_workspace):
        if force:
            shutil.rmtree(LOCAL_WORKSPACE)
            LOCAL_WORKSPACE.mkdir(parents=True, exist_ok=True)
        _copy_tree(DRIVE_WORKSPACE, LOCAL_WORKSPACE, WORKSPACE_EXCLUDED_DIRS)
        print(f"Workspace restored from: {DRIVE_WORKSPACE}")
    return state_source

restore()


## 4. Hermes AgentとOllamaの導入

Hermesは検証対象バージョンを固定します。OllamaはGemma 4に対応する最新版を公式インストーラーから導入し、
ローカルホストだけで待ち受けます。再実行時は既存インストールと既存サーバーを再利用します。


In [ ]:
def _package_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

installed_hermes = _package_version("hermes-agent")
if installed_hermes != HERMES_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"hermes-agent[all]=={HERMES_VERSION}"],
        check=True,
    )

# Colab ships an unrelated package named "cli" that shadows Hermes' top-level
# cli.py. The launcher preloads Hermes' module into sys.modules["cli"] before
# importing hermes_cli.main, making the resolution deterministic.
hermes_distribution = importlib.metadata.distribution("hermes-agent")
real_cli_path = Path(hermes_distribution.locate_file("cli.py")).resolve()
if not real_cli_path.is_file():
    raise RuntimeError(f"Hermes cli.py was not found: {real_cli_path}")

HERMES_IMPORT_SHIM = Path("/content/hermes-import-shim")
HERMES_IMPORT_SHIM.mkdir(parents=True, exist_ok=True)
HERMES_LAUNCHER = HERMES_IMPORT_SHIM / "hermes_entry.py"
launcher_source = f"""import importlib.util
import sys

_REAL_CLI = {str(real_cli_path)!r}
_SPEC = importlib.util.spec_from_file_location("cli", _REAL_CLI)
if _SPEC is None or _SPEC.loader is None:
    raise ImportError(f"Could not load Hermes cli.py: {{_REAL_CLI}}")
_MODULE = importlib.util.module_from_spec(_SPEC)
sys.modules["cli"] = _MODULE
_SPEC.loader.exec_module(_MODULE)

from hermes_cli.main import main as hermes_main
raise SystemExit(hermes_main())
"""
HERMES_LAUNCHER.write_text(launcher_source, encoding="utf-8")

if shutil.which("ollama") is None:
    # The current Ollama Linux archive is zstd-compressed. Colab images do not
    # always include the zstd executable, so install it before the official installer.
    if shutil.which("zstd") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)

    installer_url = "https://ollama.com/install.sh"
    installer_path = Path("/tmp/ollama-install.sh")
    with urllib.request.urlopen(installer_url, timeout=60) as response:
        installer_path.write_bytes(response.read())
    installer_result = subprocess.run(
        ["bash", str(installer_path)],
        text=True,
        capture_output=True,
    )
    if installer_result.returncode != 0:
        raise RuntimeError(
            "Ollamaのインストールに失敗しました。\n"
            f"stdout:\n{installer_result.stdout[-4000:]}\n"
            f"stderr:\n{installer_result.stderr[-4000:]}"
        )

def _ollama_json(path: str, timeout: int = 5) -> dict:
    with urllib.request.urlopen(f"{OLLAMA_BASE_URL}{path}", timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def _ollama_ready() -> bool:
    try:
        _ollama_json("/api/tags")
        return True
    except (OSError, urllib.error.URLError, json.JSONDecodeError):
        return False

OLLAMA_PROCESS = globals().get("OLLAMA_PROCESS")
if not _ollama_ready():
    ollama_environment = os.environ.copy()
    log_handle = OLLAMA_LOG.open("a", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=ollama_environment,
        start_new_session=True,
    )
    for _ in range(60):
        if _ollama_ready():
            break
        if OLLAMA_PROCESS.poll() is not None:
            tail = OLLAMA_LOG.read_text(encoding="utf-8", errors="replace")[-4000:]
            raise RuntimeError(f"Ollamaの起動に失敗しました。\n{tail}")
        time.sleep(1)
    else:
        raise TimeoutError(f"Ollamaが60秒以内に起動しませんでした。ログ: {OLLAMA_LOG}")

CONFIG_YAML = f"""model:
  default: "{MODEL}"
  provider: "custom"
  base_url: "{OLLAMA_BASE_URL}/v1"
  context_length: {OLLAMA_CONTEXT_LENGTH}

terminal:
  cwd: "{LOCAL_WORKSPACE}"

approvals:
  mode: "manual"
"""

if not CONFIG_FILE.exists():
    CONFIG_FILE.write_text(CONFIG_YAML, encoding="utf-8")

seed_files = {
    "MEMORY.md": "# MEMORY\n\nHermesが長期的に保持する記憶をここに蓄積します。\n",
    "USER.md": "# USER\n\nユーザーについて長期的に保持する情報をここに蓄積します。\n",
}
for filename, initial_content in seed_files.items():
    path = LOCAL_HOME / filename
    if not path.exists():
        path.write_text(initial_content, encoding="utf-8")

print(f"Hermes Agent: {_package_version('hermes-agent')}")
print(f"Ollama:       {_run_text(['ollama', '--version'])}")
print(f"Config:       {CONFIG_FILE}")
print("Ollama server: ready")


## 5. `gemma4:31b`の取得

初回は約20GBを取得します。モデルはDriveへ保存しないため、新しいColab VMでは再取得が必要です。
セルを中断せず、完了後にモデル一覧とHermes設定診断を確認してください。


In [ ]:
available_models = {
    item.get("name", "") for item in _ollama_json("/api/tags", timeout=15).get("models", [])
}
if MODEL not in available_models and f"{MODEL}:latest" not in available_models:
    subprocess.run(["ollama", "pull", MODEL], check=True)

subprocess.run(["ollama", "show", MODEL], check=True)
doctor = subprocess.run(
    ["hermes", "doctor"],
    cwd=LOCAL_WORKSPACE,
    env=os.environ.copy(),
    text=True,
    capture_output=True,
)
print(doctor.stdout)
if doctor.returncode != 0:
    print(doctor.stderr, file=sys.stderr)
    raise RuntimeError("hermes doctor が失敗しました。上の診断結果を確認してください。")

checkpoint()
print(f"{MODEL} is ready.")


## 6. Notebook操作API

- `ask_hermes(...)`: 会話を実行し、成功・失敗・中断にかかわらず最後に保存
- `new_session()`: 次の質問から新しい会話を開始
- `checkpoint()`: 手動保存
- `restore(force=True)`: Driveの正常な状態でローカルを置換
- `status()`: GPU・Ollama・モデル・Hermes・現在セッションを表示


In [ ]:
@dataclass(frozen=True)
class HermesResult:
    output: str
    returncode: int
    session_id: str | None

def _load_runtime_state() -> dict:
    if not RUNTIME_STATE_FILE.exists():
        return {"current_session_id": None, "last_checkpoint": None}
    try:
        return json.loads(RUNTIME_STATE_FILE.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {"current_session_id": None, "last_checkpoint": None}

def _save_runtime_state() -> None:
    RUNTIME_STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    temporary = RUNTIME_STATE_FILE.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(
            {
                "current_session_id": CURRENT_SESSION_ID,
                "last_checkpoint": datetime.now(timezone.utc).isoformat(),
            },
            ensure_ascii=False,
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )
    temporary.replace(RUNTIME_STATE_FILE)

_runtime_state = _load_runtime_state()
CURRENT_SESSION_ID = _runtime_state.get("current_session_id")
ACTIVE_HERMES_PROCESS: subprocess.Popen[str] | None = None

def new_session() -> None:
    """Start a new conversation on the next ask_hermes call."""
    global CURRENT_SESSION_ID
    CURRENT_SESSION_ID = None
    _save_runtime_state()
    checkpoint()
    print("New session selected.")

def _stop_process(process: subprocess.Popen[str], grace_seconds: float = 5.0) -> None:
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=grace_seconds)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

def ask_hermes(
    prompt: str,
    session_id: str | None = None,
    allow_dangerous: bool = False,
) -> HermesResult:
    """Run one Hermes turn and checkpoint state/workspace in a finally block."""
    global ACTIVE_HERMES_PROCESS, CURRENT_SESSION_ID
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("promptには空でない文字列を指定してください。")
    if not _ollama_ready():
        raise RuntimeError("Ollamaが起動していません。セットアップセルを再実行してください。")

    selected_session = session_id or CURRENT_SESSION_ID
    command = [sys.executable, str(HERMES_LAUNCHER), "chat", "-Q", "-q", prompt]
    if selected_session:
        command.extend(["--resume", selected_session])
    if allow_dangerous:
        command.append("--yolo")

    output_lines: list[str] = []
    discovered_session: str | None = selected_session
    returncode = 1
    try:
        ACTIVE_HERMES_PROCESS = subprocess.Popen(
            command,
            cwd=LOCAL_WORKSPACE,
            env=os.environ.copy(),
            stdin=subprocess.DEVNULL,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )
        assert ACTIVE_HERMES_PROCESS.stdout is not None
        for line in ACTIVE_HERMES_PROCESS.stdout:
            print(line, end="")
            output_lines.append(line)
            match = re.search(r"session_id:\s*([^\s]+)", line)
            if match:
                discovered_session = match.group(1)
        returncode = ACTIVE_HERMES_PROCESS.wait()
        if discovered_session:
            CURRENT_SESSION_ID = discovered_session
        _save_runtime_state()
        if returncode != 0:
            raise RuntimeError(f"Hermesが終了コード{returncode}で失敗しました。")
        return HermesResult("".join(output_lines), returncode, discovered_session)
    except KeyboardInterrupt:
        if ACTIVE_HERMES_PROCESS is not None:
            _stop_process(ACTIVE_HERMES_PROCESS)
        print("\nHermesを中断しました。保存可能な状態をチェックポイントします。")
        raise
    finally:
        ACTIVE_HERMES_PROCESS = None
        _save_runtime_state()
        checkpoint()

def status() -> dict:
    """Print and return a compact runtime status report."""
    runtime = _load_runtime_state()
    try:
        models = [item.get("name") for item in _ollama_json("/api/tags", timeout=5).get("models", [])]
    except Exception:
        models = []
    report = {
        "gpu": gpu_name,
        "gpu_vram_gib": round(gpu_memory_mib / 1024, 1),
        "ram_gib": round(ram_gib, 1),
        "hermes_version": _package_version("hermes-agent"),
        "ollama_ready": _ollama_ready(),
        "model_present": any(name and name.startswith(MODEL) for name in models),
        "current_session_id": CURRENT_SESSION_ID,
        "last_checkpoint": runtime.get("last_checkpoint"),
        "drive_root": str(DRIVE_ROOT),
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report

def _cleanup_active_process() -> None:
    if ACTIVE_HERMES_PROCESS is not None:
        _stop_process(ACTIVE_HERMES_PROCESS)

atexit.register(_cleanup_active_process)

import shlex
from IPython.core.error import UsageError
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def llm(line: str, cell: str | None = None) -> None:
    """Run Hermes from %llm or %%llm and store the result in LAST_HERMES_RESULT."""
    tokens = shlex.split(line)
    allow_dangerous = False
    start_new_session = False

    while tokens and tokens[0] in {"--dangerous", "--new"}:
        option = tokens.pop(0)
        if option == "--dangerous":
            allow_dangerous = True
        elif option == "--new":
            start_new_session = True

    if tokens and tokens[0].startswith("--"):
        raise UsageError(f"Unknown %llm option: {tokens[0]}")
    if cell is not None and tokens:
        raise UsageError("With %%llm, put only --new/--dangerous on the magic line.")

    prompt = cell.strip() if cell is not None else " ".join(tokens).strip()
    if not prompt:
        raise UsageError("Usage: %llm instruction  or  %%llm followed by the instruction body")

    if start_new_session:
        new_session()

    result = ask_hermes(prompt, allow_dangerous=allow_dangerous)
    get_ipython().user_ns["LAST_HERMES_RESULT"] = result
    return None

status()
print("Magic ready: %llm instruction / %%llm + multiline instruction")


## 7. 動作確認

最初の呼び出しで新しいセッションを作り、セッションID・SQLite・workspaceをDriveへ保存します。
危険操作は許可しません。


In [ ]:
result = ask_hermes(
    "日本語で一文だけ自己紹介し、現在の作業ディレクトリ名を答えてください。",
    allow_dangerous=False,
)
print(f"\nSaved session: {result.session_id}")


## 8. 継続利用

同じセッションを自動継続するため、通常は次のmagicだけを使います。

1行の指示:

```text
%llm 日本語で今日の作業方針を提案してください。
```

複数行の指示は、セル先頭を **`%%llm`** にします（`%`が2個です）。

```text
%%llm
workspaceの内容を確認してください。
その後、次に行うべき作業を3件提案してください。
```

オプション:

- `--new`: 新規セッションで開始
- `--dangerous`: その呼び出しだけ危険操作を許可
- 最後の戻り値は `LAST_HERMES_RESULT` に保存


In [ ]:
%%llm
現在のセッションが継続されていることを、日本語で一文だけ説明してください。


## 演習・運用上の注意

**演習:** `MyDrive/Hermes/state/SOUL.md` またはローカルの
`/content/hermes-home/SOUL.md` を編集し、`checkpoint()` 後に新しいランタイムから復元できることを確認してください。

**よくある問題:**

- T4/L4等が割り当てられた場合はハードウェア検査で停止します。A100へ変更してください。
- ランタイム切断中のターンは保存されない場合があります。重要な変更後は `checkpoint()` の完了表示を確認してください。
- `.env`、ログ、キャッシュ、Ollamaモデルは意図的にDrive同期対象外です。将来APIキーを使う場合はColab Secretsを使ってください。
- Driveの状態へ戻す場合は、実行中のHermesがないことを確認して `restore(force=True)` を呼びます。
